<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 2.1：你的第一个 Chisel 模块
**上一步：[Scala 简介](1_intro_to_scala.ipynb)**<br>
**下一步：[组合逻辑](2.2_comb_logic.ipynb)**

## 动机
现在你熟悉了 Scala，让我们开始构建一些硬件吧！Chisel 代表 **C**onstructing **H**ardware **I**n a **S**cala **E**mbedded **L**anguage（在 Scala 嵌入式语言中构建硬件）。这意味着它是 Scala 中的一个 DSL，允许你在同一个代码中同时利用 Scala 和 Chisel 编程。理解哪些代码是“Scala”代码，哪些是“Chisel”代码非常重要，但我们稍后会详细讨论。现在，可以将 Chisel 和模块 2 中的代码视为编写 Verilog 的更好方法。本模块向你展示了一个完整的 Chisel `Module` 和测试器。暂时只需了解其要点即可。稍后你会看到更多示例。

## 设置
以下单元格下载 Chisel 所需的依赖项。你将在所有未来的笔记本中看到它。**立即运行此单元格**。

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

正如上一个模块中提到的，这些语句是导入 Chisel 所必需的。在运行任何未来的代码块之前，**立即运行此单元格**。

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test
import dotvisualizer._

---
# 你的第一个模块
本节将介绍你的第一个硬件模块、一个测试用例以及如何运行它。它会包含许多你不会理解的内容，这没关系。我们希望你掌握大致的思路，这样你就可以不断回顾这个完整且可工作的示例，以巩固所学知识。

<span style="color:blue">**示例：一个模块**</span><br>
与 Verilog 类似，我们可以在 Chisel 中声明模块定义。以下示例是一个 Chisel `Module`，名为 `Passthrough`，它有一个 4 位输入 `in` 和一个 4 位输出 `out`。该模块将 `in` 和 `out` 进行组合连接，因此 `in` 驱动 `out`。

In [ ]:
// Chisel 代码：声明一个新的模块定义
class Passthrough extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(4.W))
    val out = Output(UInt(4.W))
  })
  io.out := io.in
}

这里有很多内容！下面解释了如何从我们描述的硬件角度理解每一行。

```scala
class Passthrough extends Module {
```
我们声明一个名为 `Passthrough` 的新模块。`Module` 是一个内置的 Chisel 类，所有硬件模块都必须扩展它。

```scala 
val io = IO(...)
```
我们在一个特殊的 `io` `val` 中声明我们所有的输入和输出端口。它必须命名为 `io` 并且是一个 `IO` 对象或实例，这需要类似 `IO(_instantiated_bundle_)` 的形式。

```scala
new Bundle {
    val in = Input(...)
    val out = Output(...)
}
```
我们声明一个新的硬件结构类型 (Bundle)，它包含一些名为 `in` 和 `out` 的信号，方向分别为 Input 和 Output。

```scala
UInt(4.W)
```
我们声明一个信号的硬件类型。在这种情况下，它是一个宽度为 4 的无符号整数。

```scala
io.out := io.in
```
我们将输入端口连接到输出端口，使得 `io.in` *驱动* `io.out`。注意 `:=` 运算符是一个 ***Chisel*** 运算符，表示右侧信号驱动左侧信号。它是一个有方向的运算符。

硬件构建语言 (HCL) 的巧妙之处在于我们可以使用底层编程语言作为脚本语言。例如，在声明我们的 Chisel 模块之后，我们然后使用 Scala 调用 Chisel 编译器将 Chisel `Passthrough` 转换为 Verilog `Passthrough`。这个过程称为***细化***。

In [ ]:
// Scala 代码：通过将其转换为 Verilog 来细化我们的 Chisel 设计
// 不用担心理解这段代码；这是非常复杂的 Scala 代码
println(getVerilog(new Passthrough))

<span style="color:blue">**示例：一个模块生成器**</span><br>
如果我们将从 Scala 中学到的知识应用到这个例子中，我们可以看到 Chisel 模块是作为 Scala 类实现的。就像任何其他 Scala 类一样，我们可以让 Chisel 模块接受一些构造参数。在这种情况下，我们创建一个新类 `PassthroughGenerator`，它将接受一个整数 `width`，该整数决定其输入和输出端口的宽度：

In [ ]:
// Chisel 代码，但传入一个参数来设置端口宽度
class PassthroughGenerator(width: Int) extends Module { 
  val io = IO(new Bundle {
    val in = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  io.out := io.in
}

// 现在让我们生成不同宽度的模块
println(getVerilog(new PassthroughGenerator(10)))
println(getVerilog(new PassthroughGenerator(20)))

请注意，生成的 Verilog 根据分配给 `width` 参数的值使用不同的输入/输出位宽。让我们深入研究一下它是如何工作的。因为 Chisel 模块是普通的 Scala 类，所以我们可以利用 Scala 类构造函数的强大功能来参数化我们设计的细化过程。

您可能会注意到这种参数化是由 *Scala* 而不是 *Chisel* 实现的；Chisel 没有用于参数化的额外 API，但设计人员可以简单地利用 Scala 功能来参数化其设计。

因为 `PassthroughGenerator` 不再描述单个模块，而是描述由 `width` 参数化的一系列模块，所以我们将此 `Passthrough` 称为***生成器***。

---
# 测试您的硬件

任何硬件模块或生成器都应该配有测试器。Chisel 内置了测试功能，您将在本训练营中进行探索。以下示例是一个 Chisel 测试工具，它将值传递给 `Passthrough` 实例的输入端口 `in`，并检查输出端口 `out` 上是否看到相同的值。

<span style="color:blue">**示例：一个测试器**</span><br>
这里有一些高级的 Scala 代码。但是，您无需理解除 `poke` 和 `expect` 命令之外的任何内容。您可以将其余代码简单地视为编写这些简单测试的样板代码。

In [ ]:
// Scala 代码：`test` 运行单元测试。
// test 接受一个用户模块，并有一个代码块，将 poke 和 expect 应用于
// 被测电路 (c)
test(new Passthrough()) { c =>
    c.io.in.poke(0.U)     // 将我们的输入设置为值 0
    c.io.out.expect(0.U)  // 断言输出正确地为 0
    c.io.in.poke(1.U)     // 将我们的输入设置为值 1
    c.io.out.expect(1.U)  // 断言输出正确地为 1
    c.io.in.poke(2.U)     // 将我们的输入设置为值 2
    c.io.out.expect(2.U)  // 断言输出正确地为 2
}
println("成功！！") // Scala 代码：如果我们到达这里，我们的测试通过了！


发生了什么？测试接受一个 `Passthrough` 模块，为模块的输入赋值，并检查其输出。要设置输入，我们调用 `poke`。要检查输出，我们调用 `expect`。如果我们不想将输出与期望值进行比较（无断言），我们可以改为 `peek` 输出。

如果所有 `expect` 语句都为真，那么我们的样板代码将返回通过。

>请注意，`poke` 和 `expect` 使用 chisel 硬件文字表示法。两个操作都期望正确类型的文字。
如果 `poke` 一个 `UInt()`，您必须提供一个 `UInt` 文字（例如：`c.io.in.poke(10.U)`），同样，如果输入是 `Bool()`，`poke` 将期望 `true.B` 或 `false.B`。



<span style="color:red">**练习：编写您自己的测试器**</span><br>
编写并执行两个测试，一个测试宽度为 10 的 `PassthroughGenerator`，另一个测试宽度为 20 的 `PassthroughGenerator`。为每个测试至少检查两个值：零和指定宽度支持的最大值。请注意，三个问号在 Scala 中具有特殊含义。您可能会在这些训练营练习中经常看到它。运行带有 `???` 的代码将产生 `NotImplementedError`。请将 `???` 替换为您自己的代码。

In [ ]:
// 测试宽度为 10

test(???) { c =>
    ???
}

// 测试宽度为 20

test(???) { c =>
    ???
}

println("成功！！") // Scala 代码：如果我们到达这里，我们的测试通过了！

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong>（点击切换显示）</label>
<article>
<pre style="background-color:#f7f7f7">
test(new PassthroughGenerator(10)) { c =>
    c.io.in.poke(0.U)
    c.io.out.expect(0.U)
    c.io.in.poke(1023.U)
    c.io.out.expect(1023.U)
}

test(new PassthroughGenerator(20)) { c =>
    c.io.in.poke(0.U)
    c.io.out.expect(0.U)
    c.io.in.poke(1048575.U)
    c.io.out.expect(1048575.U)
}

</pre></article></div></section></div>

---
# 查看生成的 Verilog/FIRRTL

如果您在理解生成的硬件时遇到困难，并且熟悉阅读结构化 Verilog 和/或 FIRRTL（Chisel 的 IR，可与 Verilog 的仅综合子集相媲美），那么您可以尝试查看生成的 Verilog 以了解 Chisel 执行的结果。

这是一个生成 Verilog（您已经看过）和 FIRRTL 的示例。

In [ ]:
// 查看 Verilog 以进行调试
println(getVerilog(new Passthrough))

In [ ]:
// 查看 firrtl 以进行调试
println(getFirrtl(new Passthrough))

---
# 您已完成！

[返回顶部。](#top)

## <span style="color:red"> 附录：关于“printf”调试的说明</span>
[使用打印语句进行调试](https://stackoverflow.com/a/189570) 并非总是最佳的调试方法，但当某些事情未按预期工作时，它通常是了解情况的简单第一步。
由于 Chisel 生成器是生成硬件的程序，因此打印生成器和电路状态存在一些额外的微妙之处。
记住打印语句何时执行以及打印什么内容非常重要。
您可能想要打印的三种常见情况具有一些重要区别：
* 电路生成期间的 Chisel 生成器打印
* 电路仿真期间的电路打印
* 测试期间的测试器打印

`println` 是一个内置的 Scala 函数，可打印到控制台。它**不能**用于在电路仿真期间打印，因为生成的电路是 FIRRTL 或 Verilog，而不是 Scala。

以下代码块显示了不同的打印样式。

In [ ]:
class PrintingModule extends Module {
    val io = IO(new Bundle {
        val in = Input(UInt(4.W))
        val out = Output(UInt(4.W))
    })
    io.out := io.in

    printf("仿真期间打印：输入为 %d\n", io.in)
    // chisel printf 也有自己的字符串插值器
    printf(p"仿真期间打印：IO 为 $io\n")

    println(s"生成期间打印：输入为 ${io.in}")
}

test(new PrintingModule ) { c =>
    c.io.in.poke(3.U)
    c.clock.step(5) // 电路将打印
    
    println(s"测试期间打印：输入为 ${c.io.in.peek()}")
}